In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import requests
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
# #data source 1: list of universities and their domains
# url = "https://raw.githubusercontent.com/Hipo/university-domains-list/master/world_universities_and_domains.json"
# response = requests.get(url)
# universities_data = json.loads(response.text)
# universities_df = pd.DataFrame(universities_data)

# #filter to only include universities in the European Union
# eu_countries = [
#     "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
#     "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
#     "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
#     "Malta", "Netherlands", "Poland", "Portugal", "Romania",
#     "Slovakia", "Slovenia", "Spain", "Sweden"
# ]
# eu_universities_df = universities_df[universities_df['country'].isin(eu_countries)]
# eu_universities_df.reset_index(drop=True, inplace=True)

# #output the dataframe to a csv file
# eu_universities_df.to_csv('eu_universities.csv', index=False)

In [3]:
# data source 2: university enrollment statistics
enrollment_df = pd.read_csv('../datafiles/eu_uni_enrollment.csv', sep=';')

In [10]:
#data source 3: university budget statistics
df = pd.read_excel('../datafiles/University_budget_stats.xlsx', header=None, engine='openpyxl')
df = df[0].str.split(';', expand=True)
df.columns = df.iloc[0]
df = df.drop(0).reset_index(drop=True)
df.replace('m', pd.NA, inplace=True)

df = df.rename(columns={
    'BAS.INSTNAME': 'name_local',
    'GEO.CITY': 'city',
    'REV.STUDFEES.EURO': 'student_fees',
    'REV.TUITFEES': 'charges_fees',
    'STUD.HIGHDEG': 'highest_degree',
    'PERS.TOTALFTE': 'staff_fte',
    'EXP.CURRTOTAL.EURO': 'total_expenditure',
})

data_cols = ['student_fees', 'charges_fees', 'highest_degree', 'staff_fte']
model_df = df[["BAS.INSTNAMEENGL", 'city'] + data_cols].copy()
model_df = model_df[model_df['BAS.INSTNAMEENGL'].notna() & (model_df['BAS.INSTNAMEENGL'].str.strip() != '')]
model_df = model_df.rename(columns={'BAS.INSTNAMEENGL': 'name'})

for col in ['student_fees', 'highest_degree', 'staff_fte']:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')
model_df['charges_fees'] = pd.to_numeric(model_df['charges_fees'], errors='coerce').fillna(0).astype(int)
model_df.reset_index(drop=True, inplace=True)

# merge web pages
uni_df = pd.read_csv('../datafiles/eu_universities.csv')
uni_merge = uni_df[['name', 'web_pages']].drop_duplicates(subset='name')
model_df = pd.merge(model_df, uni_merge, on='name', how='left')

# merge enrollment
enrollment_df = pd.read_csv('../datafiles/eu_uni_enrollment.csv', sep=';')
enrollment_df = enrollment_df.rename(columns={
    'BAS.INSTNAMEENGL': 'name',
    'STUD.TOTALISCED5-7': 'total_students'
})
enrollment_df['total_students'] = pd.to_numeric(enrollment_df['total_students'], errors='coerce')
model_df = pd.merge(model_df, enrollment_df[['name', 'total_students']], on='name', how='left')

# calculate per student fees
model_df['per_student_fees'] = np.where(
    model_df['charges_fees'] == 0,
    0.0,
    model_df['student_fees'] / model_df['total_students']
)

# merge coordinates
coords_df = pd.read_csv('../datafiles/model_df_with_coords.csv')[['name', 'latitude', 'longitude']]
model_df = pd.merge(model_df, coords_df, on='name', how='left')

model_df = model_df.drop_duplicates(subset='name').reset_index(drop=True)
model_df.to_csv('../datafiles/model_df.csv', index=False)
print(model_df.shape)
print(model_df[['per_student_fees', 'highest_degree', 'staff_fte']].describe())

(780, 11)
       per_student_fees  highest_degree     staff_fte
count        704.000000      764.000000    608.000000
mean         101.731580        2.473822   1200.255325
std          379.353898        0.652123   1751.255280
min            0.000000        0.000000      8.000000
25%            0.000000        2.000000    148.270767
50%            0.000000        3.000000    496.604867
75%            0.000000        3.000000   1367.500000
max         4517.270412        3.000000  12350.110000


In [ ]:
model_cols = ['per_student_fees', 'highest_degree', 'staff_fte']

X_mat = model_df[model_cols].dropna()
model_df_clean = model_df.loc[X_mat.index].reset_index(drop=True)
X_mat = X_mat.reset_index(drop=True).to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_mat)
staff_min = model_df_clean['staff_fte'].min()
staff_max = model_df_clean['staff_fte'].max()

# student survey inputs
student_budget = 2000
student_degree = 3
student_size = 2

student_staff = staff_min + (student_size - 1) * (staff_max - staff_min) / 2
student_input = np.array([[student_budget, student_degree, student_staff]])
student_scaled = scaler.transform(student_input)

cosine_scores = []
for i in range(X_scaled.shape[0]):
    uni_vec = X_scaled[i]
    student_vec = student_scaled[0]
    dot = np.dot(student_vec, uni_vec)
    student_norm = np.linalg.norm(student_vec)
    uni_norm = np.linalg.norm(uni_vec)
    cos = dot / (student_norm * uni_norm) if student_norm != 0 and uni_norm != 0 else 0
    cosine_scores.append(cos)

results_df = pd.DataFrame({
    'name':         model_df_clean['name'],
    'city':         model_df_clean['city'],
    'cosine_score': cosine_scores
})

ranked = results_df.sort_values(by='cosine_score', ascending=False).reset_index(drop=True)
ranked.index = ranked.index + 1
ranked['match_number'] = (ranked['cosine_score'] * 100).round(2).astype(str)

output = ranked[['name', 'city', 'match_number']]
output.index.name = 'rank'

print("TOP 10 MATCHES")
print(output.head(10).to_string())
print("\n... view more ...")
print(output.tail().to_string())

TOP 10 MATCHES
                                           name                city match_number
rank                                                                            
1                         University of Antwerp             Antwerp        99.76
2                The Vrije Universiteit Brussel             Ixelles        99.63
3                           University of Tartu               Tartu        98.29
4                University of Southern Denmark              Odense        96.19
5                            Aalborg University             Aalborg        96.19
6              Tallinn University of Technology             Tallinn        95.48
7                    Copenhagen Business School       Frederiksberg        95.48
8               Technical University of Denmark              Lyngby        95.13
9                            Hasselt University          Diepenbeek        95.11
10    University for Continuing Education Krems  Krems an der Donau        92.99

... view mor